# Encoding gauge: trainability is invariant, with the scalable DLA floor

Companion computation to the encoding-gauge study. Self-validating: the engine and the gradient
invariance are checked against independent ground truth before the study runs. Part 1 computes the
dynamical-Lie-algebra dimension by a sparse Lie-closure over Pauli strings (no exponential matrices),
giving the floor at scale. Part 2 measures the gradient variance, showing it is identical across
encodings and follows the inverse-dimension law.


In [ ]:
!pip install -q openfermion joblib

In [ ]:
# ============================ validated engine (inlined verbatim) ============================
import numpy as np, os, math
import openfermion as of
from openfermion import (FermionOperator, jordan_wigner, bravyi_kitaev, bravyi_kitaev_tree,
                         binary_code_transform, parity_code, get_sparse_operator, hermitian_conjugated)
from joblib import Parallel, delayed

def _ofpatch():
    import openfermion.ops.operators.symbolic_operator as _s
    if not isinstance(np.int64(0), _s.COEFFICIENT_TYPES):
        _s.COEFFICIENT_TYPES = tuple(_s.COEFFICIENT_TYPES) + (np.integer, np.floating, np.complexfloating)
_ofpatch()
ENC = ["JW", "BK", "parity"]

def _pc(x): return int(x).bit_count()

# ---- Pauli propagation (validated vs statevector to 1e-15) ----
def _commute(xp, zp, xg, zg): return (_pc(xp & zg) + _pc(zp & xg)) % 2 == 0
def _pmul(xp, zp, xg, zg): return xp ^ xg, zp ^ zg, (-1.0 if (_pc(zp & xg) & 1) else 1.0)
def _conj_gate(terms, xg, zg, theta):
    c = np.cos(2 * theta); s = np.sin(2 * theta); gamma = (1j) ** _pc(xg & zg); out = {}
    for (xp, zp), coeff in terms.items():
        if _commute(xp, zp, xg, zg):
            out[(xp, zp)] = out.get((xp, zp), 0j) + coeff
        else:
            out[(xp, zp)] = out.get((xp, zp), 0j) + coeff * c
            xr, zr, sg = _pmul(xp, zp, xg, zg); out[(xr, zr)] = out.get((xr, zr), 0j) + coeff * (-1j) * s * gamma * sg
    return out
def _truncate(terms, w_max, delta):
    if w_max is None and delta <= 0: return terms
    return {(x, z): c for (x, z), c in terms.items() if abs(c) >= max(delta, 1e-300) and (w_max is None or _pc(x | z) <= w_max)}
def propagate_circuit(n, gates, obs, w_max=None, delta=0.0):
    terms = dict(obs)
    for (xg, zg, th) in reversed(gates):
        terms = _truncate(_conj_gate(terms, xg, zg, th), w_max, delta)
    return _truncate(terms, w_max, delta)

# ---- statevector ground truth ----
def _pauli_apply(state, x, z, n):
    k = np.arange(1 << n); sign = np.ones(1 << n); zz = z
    while zz:
        b = (zz & -zz).bit_length() - 1; sign = sign * (1 - 2 * ((k >> b) & 1)); zz &= zz - 1
    return (state * sign)[k ^ x]
def apply_rotation(state, x, z, theta, n):
    return np.cos(theta) * state - 1j * np.sin(theta) * ((1j) ** _pc(x & z) * _pauli_apply(state, x, z, n))

# ---- encodings + helpers ----
def encode(fop, enc, n):
    if enc == "JW": return jordan_wigner(fop)
    if enc == "BK": return bravyi_kitaev(fop, n_qubits=n)
    if enc == "BKtree": return bravyi_kitaev_tree(fop, n_qubits=n)
    if enc == "parity":
        _ofpatch(); return binary_code_transform(fop, parity_code(n))
    raise ValueError(enc)
def dense(q, n): return np.asarray(get_sparse_operator(q, n_qubits=n).todense())
def qop_to_terms(qop):
    out = {}
    for term, coeff in qop.terms.items():
        x = z = 0
        for q, p in term:
            if p in ("X", "Y"): x |= 1 << q
            if p in ("Z", "Y"): z |= 1 << q
        out[(x, z)] = out.get((x, z), 0j) + complex(coeff)
    return {k: v.real for k, v in out.items() if abs(v) > 1e-12}
def qop_to_xz(qop):
    out = {}
    for term, coeff in qop.terms.items():
        x = z = 0
        for q, p in term:
            if p in ("X", "Y"): x |= 1 << q
            if p in ("Z", "Y"): z |= 1 << q
        out[(x, z)] = out.get((x, z), 0j) + complex(coeff) * (1j) ** _pc(x & z)
    return {k: v for k, v in out.items() if abs(v) > 1e-12}

def hf_index(enc, n, ne):
    occ = [1] * ne + [0] * (n - ne)
    nt = [qop_to_terms(encode(FermionOperator("%d^ %d" % (i, i)), enc, n)) for i in range(n)]
    for b in range(1 << n):
        if all(abs(sum(c * (1.0 if not (_pc(z & b) & 1) else -1.0) for (x, z), c in nt[i].items() if x == 0) - occ[i]) < 1e-6 for i in range(n)):
            return b
    raise RuntimeError("HF")
def circuit(enc, gens, n, angles):
    g = []
    for layer in angles:
        for th, gen in zip(layer, gens):
            for (x, z), c in qop_to_terms(encode(gen, enc, n)).items():
                if (x, z) != (0, 0): g.append((x, z, th * c))
    return g
def exact_O(n, gates, obs, b):
    psi = np.zeros(1 << n, dtype=complex); psi[b] = 1.0
    for (x, z, th) in gates: psi = apply_rotation(psi, x, z, th, n)
    return sum(a * ((1j) ** _pc(x & z)) * np.vdot(psi, _pauli_apply(psi, x, z, n)) for (x, z), a in obs.items()).real
def exp_basis(terms, b):
    return float(sum(np.real(c) * (1.0 if not (_pc(z & b) & 1) else -1.0) for (x, z), c in terms.items() if x == 0))

# ---- magic ----
def two_sre(psi, n):
    d = 1 << n; idx = np.arange(d); pc = np.array([bin(b).count("1") for b in range(d)]); s2 = s4 = 0.0
    for x in range(d):
        cps = np.conjugate(psi[idx ^ x])
        for z in range(d):
            e = ((1j) ** (bin(x & z).count("1")) * (cps @ ((1.0 - 2.0 * (pc[idx & z] & 1)) * psi))).real
            s2 += e * e; s4 += e ** 4
    assert abs(s2 - d) < 1e-6 * d
    return -np.log2(s4 / d)

# ---- DLA: dense (validation) and sparse-over-Paulis (scalable) ----
def dla_dim(mats, cap=800):
    def vec(A): return np.concatenate([A.real.ravel(), A.imag.ravel()]).astype(float)
    basis, Q, queue = [], [], []
    def add(A):
        v = vec(A)
        for q in Q: v = v - (q @ v) * q
        if np.linalg.norm(v) > 1e-7: Q.append(v / np.linalg.norm(v)); basis.append(A); return True
        return False
    for M in mats:
        if add(1j * M): queue.append(basis[-1])
    while queue:
        A = queue.pop()
        for B in list(basis):
            C = A @ B - B @ A
            if np.linalg.norm(C) < 1e-9: continue
            if add(C):
                queue.append(basis[-1])
                if len(basis) >= cap: return cap
    return len(basis)
def _bracket(A, B):
    out = {}
    for (xa, za), ca in A.items():
        for (xb, zb), cb in B.items():
            if (_pc(za & xb) + _pc(zb & xa)) & 1:
                R = (xa ^ xb, za ^ zb); out[R] = out.get(R, 0j) + 2.0 * ca * cb * ((-1.0) ** _pc(za & xb))
    return {k: v for k, v in out.items() if abs(v) > 1e-12}
def _realsp(A):
    v = {}
    for k, c in A.items():
        if abs(c.real) > 1e-12: v[(k, 0)] = c.real
        if abs(c.imag) > 1e-12: v[(k, 1)] = c.imag
    return v
def dla_dim_pauli(gens_aH, cap=20000):
    piv = {}
    def reduce(v):
        v = dict(v); prog = True
        while prog:
            prog = False
            for pk, pv in piv.items():
                if abs(v.get(pk, 0.0)) > 1e-9:
                    f = v[pk] / pv[pk]
                    for c, val in pv.items():
                        nv = v.get(c, 0.0) - f * val
                        if abs(nv) < 1e-9: v.pop(c, None)
                        else: v[c] = nv
                    prog = True
        return v
    def add(A):
        v = reduce(_realsp(A))
        if not v: return False
        piv[max(v, key=lambda k: abs(v[k]))] = v; return True
    basis, queue = [], []
    for g in gens_aH:
        if add(g): basis.append(g); queue.append(g)
    while queue:
        A = queue.pop()
        for B in list(basis):
            C = _bracket(A, B)
            if C and add(C):
                basis.append(C); queue.append(C)
                if len(piv) >= cap: return cap
    return len(piv)
def aH(gens, enc, n): return [{k: 1j * c for k, c in qop_to_xz(encode(g, enc, n)).items()} for g in gens]

# ---- models + gradient machinery ----
def free_gens(n): return [FermionOperator("%d^ %d" % (i, j)) + FermionOperator("%d^ %d" % (j, i)) for i in range(n) for j in range(i + 1, n)]
def interacting_gens(n):
    g = []
    for i in range(n):
        for j in range(i + 1, n):
            g.append(FermionOperator("%d^ %d" % (i, j)) + FermionOperator("%d^ %d" % (j, i)))
            g.append(FermionOperator("%d^ %d %d^ %d" % (i, i, j, j)))
    return g
def Oexp(enc, gens, n, b, obs, theta): return exact_O(n, circuit(enc, gens, n, theta), obs, b)
def sample_theta(ng, layers, rng): return [[float(rng.uniform(0, 2 * np.pi)) for _ in range(ng)] for _ in range(layers)]
def grad_vec(enc, gens, n, b, obs, theta, eps=1e-4):
    g = []
    for L in range(len(theta)):
        for j in range(len(theta[L])):
            tp = [r[:] for r in theta]; tp[L][j] += eps; tm = [r[:] for r in theta]; tm[L][j] -= eps
            g.append((Oexp(enc, gens, n, b, obs, tp) - Oexp(enc, gens, n, b, obs, tm)) / (2 * eps))
    return np.array(g)
def grad_component(enc, gens, n, b, obs, theta, k, eps=1e-4):
    ng = len(theta[0]); L, j = k // ng, k % ng
    tp = [r[:] for r in theta]; tp[L][j] += eps; tm = [r[:] for r in theta]; tm[L][j] -= eps
    return (Oexp(enc, gens, n, b, obs, tp) - Oexp(enc, gens, n, b, obs, tm)) / (2 * eps)

def var_grad(args):
    _ofpatch()
    model, enc, n, layers, S, K, seed, B = args
    gens = (free_gens if model == "free" else interacting_gens)(n); ne = n // 2
    rng = np.random.default_rng(seed)
    b = hf_index(enc, n, ne); ob = qop_to_terms(encode(FermionOperator("0^ 0"), enc, n))
    gt = grad_vec(enc, gens, n, b, ob, sample_theta(len(gens), layers, rng))
    ks = [int(k) for k in np.argsort(-np.abs(gt))[:K]]
    thetas = [sample_theta(len(gens), layers, rng) for _ in range(S)]
    Gm = np.array([[grad_component(enc, gens, n, b, ob, th, k) for k in ks] for th in thetas])  # (S,K)
    point = float(np.mean(np.var(Gm, axis=0)))                        # mean over components of Var over S samples
    brng = np.random.default_rng(987)                                 # 95% percentile bootstrap CI over the S samples
    boots = [float(np.mean(np.var(Gm[brng.integers(0, S, S)], axis=0))) for _ in range(B)]
    lo, hi = float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))
    return (model, enc, n, point, lo, hi)

def dla_point(args):
    _ofpatch()
    model, n = args
    gens = (free_gens if model == "free" else interacting_gens)(n)
    return (model, n, dla_dim_pauli(aH(gens, "JW", n), cap=4000))
print("engine loaded. CPU cores:", os.cpu_count())

## Step 1 - certify the engine and the gradient invariance

In [ ]:
# ===================== SELF-VALIDATION against independent ground truth =====================
ok = []
# engine == statevector
n = 4; gens = interacting_gens(n); ne = 2; ang = [list(np.random.default_rng(1).uniform(.3, .9, size=len(gens)))]
worst = 0.0
for e in ENC:
    b = hf_index(e, n, ne); g = circuit(e, gens, n, ang); ob = qop_to_terms(encode(FermionOperator("0^ 0"), e, n))
    worst = max(worst, abs(exp_basis(propagate_circuit(n, g, ob, w_max=n), b) - exact_O(n, g, ob, b)))
ok.append(("engine == statevector", worst < 1e-7))
# DLA dense == su(n); sparse == dense
ds = {dla_dim([dense(encode(x, e, 5), 5) for x in free_gens(5)], cap=200) for e in ENC}
ok.append(("dense DLA == su(5)=24, all encodings", ds == {24}))
ok.append(("sparse DLA == dense (free n=5, int n=4)", dla_dim_pauli(aH(free_gens(5), "JW", 5)) == 24 and dla_dim_pauli(aH(interacting_gens(4), "JW", 4)) == 66))
# magic analytic
t = np.array([1.0, np.exp(1j * np.pi / 4)]) / np.sqrt(2)
ok.append(("magic SRE: T-state==log2(4/3), stabilizer==0", abs(two_sre(t, 1) - np.log2(4 / 3)) < 1e-6 and abs(two_sre(np.array([1.0, 0.0]), 1)) < 1e-9))
# gradient invariance (the central claim of the trainability part)
gv = {e: grad_vec(e, gens, n, hf_index(e, n, ne), qop_to_terms(encode(FermionOperator("0^ 0"), e, n)),
                  [list(np.random.default_rng(2).uniform(0, 2 * np.pi, size=len(gens)))]) for e in ENC}
gdiff = max(np.max(np.abs(gv["JW"] - gv[e])) for e in ENC)
ok.append(("gradient vector identical across encodings (<1e-9)", gdiff < 1e-9 and np.max(np.abs(gv["JW"])) > 1e-3))
for name, p in ok: print("[%s] %s" % ("PASS" if p else "FAIL", name))
assert all(p for _, p in ok), "VALIDATION FAILED"
print("\n>>> all %d checks PASSED: engine + gradient invariance certified on this machine.\n" % len(ok))

## Step 2 - the floor at scale (Part 1)

In [ ]:
# ============== PART 1: scalable DLA -- dim g = su(n) = n^2-1 (poly) while JW weight = n ==============
import matplotlib.pyplot as plt
NF = list(range(4, 25, 2))
res = Parallel(n_jobs=-1)(delayed(dla_point)(("free", n)) for n in NF)
dimg = {n: d for (_, n, d) in res}
print("n   dim g   n^2-1   JW weight")
for n in NF:
    w = max((bin(x | z).count("1") for g in free_gens(n) for (x, z) in qop_to_xz(encode(g, "JW", n))), default=0)
    print("%-3d %-7d %-7d %-3d" % (n, dimg[n], n * n - 1, w))
plt.figure(figsize=(6.2, 4))
plt.plot(NF, [dimg[n] for n in NF], "o-", label="dim $\\mathfrak{g}$ (computed)")
plt.plot(NF, [n * n - 1 for n in NF], "k--", alpha=.5, label="$n^2-1=\\dim\\mathfrak{su}(n)$")
plt.plot(NF, NF, "s-", label="Jordan-Wigner Pauli weight $=n$")
plt.xlabel("n (qubits / modes)"); plt.ylabel("resource value")
plt.title("The floor at scale: invariant dim $\\mathfrak{g}$ is polynomial, covariant weight is linear")
plt.legend(); plt.tight_layout(); plt.savefig("fig_dla_scale.png", dpi=130); plt.show()

## Step 3 - gradient-variance scaling (Part 2)

In [ ]:
# ============== PART 2: gradient-variance scaling -- gauge-invariant, follows 1/dim g ==============
# Estimator: mean over the K most active gradient components of the variance over S random theta;
# 95% confidence interval by percentile bootstrap (B resamples) over the S samples.
layers, S, K, B = 2, 400, 8, 2000
print("invariance check (Var ratio JW/BK should be exactly 1):")
for model in ("free", "interacting"):
    for n in (4, 6, 8):
        vj = var_grad((model, "JW", n, layers, S, K, 11, B))[3]
        vb = var_grad((model, "BK", n, layers, S, K, 11, B))[3]
        print("  %-11s n=%d  ratio=%.6f" % (model, n, vj / vb))

NF = [4, 6, 8, 10, 12, 14, 16]
NI = [4, 6, 8, 10, 12]
tasks = [("free", "JW", n, layers, S, K, 11, B) for n in NF] + [("interacting", "JW", n, layers, S, K, 11, B) for n in NI]
out = Parallel(n_jobs=-1)(delayed(var_grad)(t) for t in tasks)
var = {(m, n): (v, lo, hi) for (m, e, n, v, lo, hi) in out}
dg = dict((n, d) for (_, n, d) in Parallel(n_jobs=-1)(delayed(dla_point)(("free", n)) for n in NF))

print("\n# S=%d random theta, K=%d components, B=%d bootstrap. Transcribe these into make_figs.py:" % (S, K, B))
print("# model        n   Var          CI_low       CI_high")
for m, ns in (("free", NF), ("interacting", NI)):
    for n in ns:
        v, lo, hi = var[(m, n)]; print("%-12s %-3d %.6e %.6e %.6e" % (m, n, v, lo, hi), flush=True)

def yerr(ns, m): return np.array([[var[(m,n)][0]-var[(m,n)][1] for n in ns], [var[(m,n)][2]-var[(m,n)][0] for n in ns]])
plt.figure(figsize=(6.2, 4))
plt.errorbar(NF, [var[("free", n)][0] for n in NF], yerr=yerr(NF,"free"), fmt="o-", capsize=3, label="free fermions (dim $\\mathfrak{g}=n^2-1$, poly)")
plt.errorbar(NI, [var[("interacting", n)][0] for n in NI], yerr=yerr(NI,"interacting"), fmt="s-", capsize=3, label="interacting (dim $\\mathfrak{g}$ exponential)")
C = float(np.mean([var[("free", n)][0] * dg[n] for n in NF]))
plt.semilogy(NF, [C / (n * n - 1) for n in NF], "k--", alpha=.6, label="$%.1f/(n^2-1)$ (the $1/\\dim\\mathfrak{g}$ law)" % C)
plt.yscale("log")
plt.xlabel("n (qubits / modes)"); plt.ylabel("Var[$\\partial_\\theta\\langle O\\rangle$]")
plt.title("Trainability is gauge-invariant: Var[grad] identical across encodings,\nfollowing the $1/\\dim\\mathfrak{g}$ law (95%% bootstrap CIs, S=%d)" % S)
plt.legend(); plt.tight_layout(); plt.savefig("fig_variance.png", dpi=130); plt.show()
print("\nfree-fermion Var*dim_g (constant => Var ~ 1/dim g):", ["%.2f" % (var[("free", n)][0] * dg[n]) for n in NF])

## Conclusion
The gradient variance is identical across encodings (ratio one) and follows the inverse of the
gauge-invariant dynamical-Lie-algebra dimension, which itself is polynomial for free fermions out to
the largest sizes computed here while the Jordan-Wigner Pauli weight is linear. Simulation cost can be
lowered by re-encoding; trainability cannot.
